# 04_extract_text

Dry-run text extraction on the latest inventory output. This notebook stays conservative: text-like files, PDF text extraction, DOCX paragraphs/tables, and XLSX sheet previews. No OCR yet.


In [2]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.extractors import ExtractConfig, enrich_inventory_with_text, save_text_outputs

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
inventory_files = sorted(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet found. Run 02_inventory.ipynb first.'
INVENTORY_PATH = inventory_files[-1]
print('Using inventory file:', INVENTORY_PATH.name)


Using inventory file: inventory_20260307_074202.parquet


In [3]:
inv = pd.read_parquet(INVENTORY_PATH)
print('Rows:', len(inv))
display(inv[['relative_path', 'suffix', 'size_bytes']].head(10))
display(inv['suffix'].fillna('').value_counts().rename_axis('suffix').reset_index(name='count').head(20))


Rows: 5


KeyError: "['suffix'] not in index"

In [ ]:
config = ExtractConfig(
    max_chars_per_file=12000,
    max_csv_rows=30,
    max_csv_columns=20,
    max_xlsx_rows_per_sheet=30,
    max_xlsx_columns=20,
    max_docx_paragraphs=300,
    max_pdf_pages=30,
    preview_chars=300,
)
config


In [ ]:
enriched = enrich_inventory_with_text(inv, path_column='absolute_path', config=config)
display(enriched[['relative_path', 'suffix', 'text_status', 'text_source', 'extracted_chars', 'text_preview']].head(20))


In [ ]:
display(enriched['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(enriched['text_source'].value_counts(dropna=False).rename_axis('text_source').reset_index(name='count'))
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']].head(20))


In [ ]:
display(enriched[enriched['has_extracted_text']][['relative_path', 'text_source', 'extracted_chars', 'text_preview']].head(20))


In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'inventory_with_text_{timestamp}'
csv_path, parquet_path = save_text_outputs(enriched, output_base)
print('Saved CSV   :', csv_path)
print('Saved Parquet:', parquet_path)
